# EVT Threshold Calibration (POT) for a HydroBASINS L12 Unit (Virtual Gauge)

This notebook is designed for **calibration** (Objective 1): selecting a statistically sound **Peaks-over-Threshold (POT)** configuration for **one HydroBASINS L12 sub-catchment** using **GloFAS historical/reanalysis discharge**.

### What this notebook does (end-to-end)
1. **Loads the basin config** (expects `hydrobasins_level=12` and a single `hydrobasins_id`).
2. Builds the **virtual gauge**:
   - reads the **HydroBASINS L12 pour point** for that `hybas_id`;
   - maps it to the **nearest GloFAS grid cell** (no averaging).
3. **Extracts a daily discharge time series from GRIB** (year-by-year files) *if the time series does not exist yet*.
4. Provides guided, visual diagnostics to choose:
   - an EVT threshold `u` (m³/s) and
   - a declustering window `run_length_days` (default 5).
5. Outputs a **copy/paste YAML snippet** for the calibrated parameters, including the Poisson frequency parameter **λ** (events/year).

### Why “virtual gauges”?
GloFAS reanalysis is **gridded**; there are no physical station IDs everywhere. We define a **representative river cell per L12** using the HydroBASINS pour point (outlet-like).  
If two L12 basins map to the *same* GloFAS cell, that is a **flag for merging or manual review** (we do *not* average).

---

## Analyst checklist (what good looks like)
- **Time series looks reasonable** (no long constant segments, no obvious unit issues).
- **MRL plot** is approximately linear above the chosen threshold.
- **GPD parameter stability** is reasonably stable across a small threshold range.
- **Event count** is sufficient (rule of thumb: at least ~50–100 exceedances over the full record for stable tail estimation, depending on basin).

> Note: This notebook focuses on **threshold + frequency λ**. Fitting the final GPD parameters and generating synthetic events happens in the next calibration notebook.


In [1]:
# Auto‑reload modules so that changes in src/ are picked up without restarting the kernel
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Optional interactivity
try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

def find_repo_root(start: Path | None = None) -> Path:
    """Find repo root by walking upwards until 'src/philflood' is found."""
    start = start or Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / 'src' / 'philflood').exists():
            return p
    raise FileNotFoundError("Could not locate repo root (expected 'src/philflood' in parents).")

REPO_ROOT = find_repo_root()
SRC_PATH = REPO_ROOT / 'src'
if SRC_PATH.as_posix() not in sys.path:
    sys.path.insert(0, SRC_PATH.as_posix())

print('Repo root:', REPO_ROOT)

# Dependency checks (fail early with actionable guidance)
missing = []
try:
    import xarray as xr
except Exception:
    missing.append('xarray')
try:
    import cfgrib  # noqa: F401
    HAS_CFGRIB = True
except Exception:
    HAS_CFGRIB = False

try:
    import geopandas as gpd
    HAS_GPD = True
except Exception:
    HAS_GPD = False

if not HAS_CFGRIB:
    warnings.warn(
        "cfgrib is not available. Install via conda-forge: 'conda install -c conda-forge cfgrib python-eccodes'.\n"
        "GRIB extraction will not work until cfgrib + ecCodes are installed."
    )

if not HAS_GPD:
    warnings.warn(
        "geopandas is not available. Install via conda-forge: 'conda install -c conda-forge geopandas'.\n"
        "HydroBASINS pour point reading will not work until geopandas is installed."
    )

# Project imports
from philflood.domain.config import load_basin_config
from philflood.adapters.glofas import load_glofas_reanalysis_for_basin

from philflood.models.ev.threshold_analysis import (
    extract_declust_pot,
    mean_residual_life,
    gpd_parameter_stability,
    suggest_threshold_range,
)

plt.rcParams.update({
    'figure.figsize': (10, 4),
    'axes.grid': True,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
})


Repo root: C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL


## Inputs

You should only need to adjust the variables in the next cell.

**Required:**
- `basin_cfg_relpath`: relative path to the basin YAML (e.g., `ops/configs/basins/Cagayan_01.yaml`)
- `glofas_grib_root`: root folder containing yearly GRIB files
- `hybas_pourpoints_shp`: HydroBASINS L12 pour points shapefile
- `start_date`, `end_date`: calibration period

**Optional:**
- `force_reextract_timeseries`: set `True` to rebuild the time series even if a cached file exists
- `candidate_quantiles`: quantiles used to propose initial thresholds (we refine using diagnostics)


In [2]:
# ---- User inputs (edit here) ----

# Basin YAML (relative to repo root)
basin_cfg_relpath = r"ops\configs\basins\Cagayan_01.yaml"

# GloFAS GRIB root (year folders underneath)
glofas_grib_root = REPO_ROOT / "data" / "raw" / "glofas" / "historical" / "version_4_0" / "consolidated" / "discharge" / "grib2" / "area_35_63_4_131"

# HydroBASINS L12 pour points shapefile
hybas_pourpoints_shp = REPO_ROOT / "data" / "raw" / "vectors" / "hydrobasins" / "australasia" / "hybas_pour_lev01-12_v1_shp" / "hybas_pour_lev12_v1.shp"

# Calibration period
start_date = "1979-01-01"
end_date   = "2023-12-31"

# EVT choices
run_length_days_default = 5
candidate_quantiles = [0.95, 0.97, 0.98, 0.99]

# Time series caching
force_reextract_timeseries = False

# Output cache location (processed data)
timeseries_out_dir = REPO_ROOT / "data" / "processed" / "glofas" / "timeseries" / "version_4_0" / "consolidated" / "area_35_63_4_131"
timeseries_out_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------

basin_cfg_path = (REPO_ROOT / basin_cfg_relpath) if not Path(basin_cfg_relpath).is_absolute() else Path(basin_cfg_relpath)

if not basin_cfg_path.exists():
    print(f"ERROR: basin config not found: {basin_cfg_path}")
    print("Available basin configs:")
    for p in sorted((REPO_ROOT / 'ops' / 'configs' / 'basins').glob('*.yaml')):
        print(" -", p.relative_to(REPO_ROOT))
    raise FileNotFoundError(basin_cfg_path)

print("Basin config:", basin_cfg_path)
print("GRIB root:", glofas_grib_root)
print("Pour points:", hybas_pourpoints_shp)


Basin config: C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\ops\configs\basins\Cagayan_01.yaml
GRIB root: C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\raw\glofas\historical\version_4_0\consolidated\discharge\grib2\area_35_63_4_131
Pour points: C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\raw\vectors\hydrobasins\australasia\hybas_pour_lev01-12_v1_shp\hybas_pour_lev12_v1.shp


In [3]:
# Load and validate basin config
basin_cfg = load_basin_config(basin_cfg_path)
issues = basin_cfg.validate() if hasattr(basin_cfg, "validate") else []
print(f"Loaded basin configuration: {basin_cfg.basin_id}")
print(f"hydrobasins_level: {basin_cfg.hydrobasins_level}")
print(f"hydrobasins_id   : {basin_cfg.hydrobasins_id}")

if issues:
    print("\nConfig validation issues:")
    for i in issues:
        print(" -", i)

if basin_cfg.hydrobasins_level != 12:
    warnings.warn("This notebook is intended for HydroBASINS L12 units. Proceeding anyway, but check your YAML.")

hybas_id = int(basin_cfg.hydrobasins_id)


Loaded basin configuration: Cagayan_01
hydrobasins_level: 7
hydrobasins_id   : 0


C:\Users\duruenaramirez\AppData\Local\Temp\ipykernel_15152\1774941881.py:14: UserWarning: This notebook is intended for HydroBASINS L12 units. Proceeding anyway, but check your YAML.
  warnings.warn("This notebook is intended for HydroBASINS L12 units. Proceeding anyway, but check your YAML.")


## Step 1 — Build the virtual gauge (one representative point per L12)

We use the **HydroBASINS L12 pour point** (outlet-like) for `hydrobasins_id` and map it to the nearest **GloFAS grid cell**.

Outputs:
- `points_df`: table with the pour point and mapped GloFAS grid coordinate
- an interactive map for QA/QC


In [ ]:
# Build the virtual gauge for this L12 (pour point → nearest GloFAS grid cell)

if not HAS_GPD:
    raise ImportError("geopandas is required to read HydroBASINS pour points.")

if not hybas_pourpoints_shp.exists():
    raise FileNotFoundError(f"Pour points shapefile not found: {hybas_pourpoints_shp}")

gdf = gpd.read_file(hybas_pourpoints_shp)

# Try common ID field names (dataset often uses 'HYBAS_ID')
id_field = 'HYBAS_ID' if 'HYBAS_ID' in gdf.columns else None
if id_field is None:
    # fallback: try case-insensitive match
    for c in gdf.columns:
        if c.lower() == 'hybas_id':
            id_field = c
            break
if id_field is None:
    raise ValueError(f"Could not find a HYBAS_ID field in pour points. Columns: {list(gdf.columns)}")

row = gdf.loc[gdf[id_field].astype(int) == hybas_id]
if row.empty:
    raise ValueError(f"hybas_id={hybas_id} not found in pour points layer.")
row = row.iloc[0]

# Pour point coordinates
pp_lon = float(row.geometry.x)
pp_lat = float(row.geometry.y)

points_df = pd.DataFrame([{
    "point_id": str(hybas_id),
    "hybas_id": hybas_id,
    "pour_lat": pp_lat,
    "pour_lon": pp_lon,
    "source": "hydrobasins_pour_point"
}])

# Find a sample GRIB file to read grid coordinates
if not glofas_grib_root.exists():
    raise FileNotFoundError(f"GRIB root does not exist: {glofas_grib_root}")

sample_files = sorted(glofas_grib_root.glob("*/*.grib*"))
if not sample_files:
    raise FileNotFoundError(f"No GRIB files found under: {glofas_grib_root}")

sample_grib = sample_files[0]
print("Sample GRIB:", sample_grib)

# Keep cfgrib index files out of raw/ (important for clean data management)
index_dir = timeseries_out_dir / "_cfgrib_index"
index_dir.mkdir(parents=True, exist_ok=True)
indexpath = str(index_dir / f"{sample_grib.stem}.idx")

def _open_grib(path: Path, indexpath: str):
    """Open GRIB robustly (handles multi-message files)."""
    try:
        return xr.open_dataset(path, engine="cfgrib", backend_kwargs={"indexpath": indexpath})
    except Exception:
        import cfgrib
        dsets = cfgrib.open_datasets(path, backend_kwargs={"indexpath": indexpath})
        # Prefer datasets with time + lat/lon
        for ds_ in dsets:
            coords = set(ds_.coords)
            if ('time' in coords or 'valid_time' in coords) and (('latitude' in coords or 'lat' in coords) and ('longitude' in coords or 'lon' in coords)):
                return ds_
        return dsets[0]

ds = _open_grib(sample_grib, indexpath=indexpath)

# Detect coordinate names
lat_name = "latitude" if "latitude" in ds.coords else ("lat" if "lat" in ds.coords else None)
lon_name = "longitude" if "longitude" in ds.coords else ("lon" if "lon" in ds.coords else None)
if lat_name is None or lon_name is None:
    raise ValueError(f"Could not find lat/lon coords in GRIB dataset. Coords: {list(ds.coords)}")

lats = ds[lat_name].values
lons = ds[lon_name].values

# Normalise longitudes if GRIB uses 0..360
pp_lon_adj = pp_lon
if np.nanmax(lons) > 180 and pp_lon_adj < 0:
    pp_lon_adj = pp_lon_adj % 360

# Find nearest grid coordinate (no averaging)
nearest_lat = float(lats[np.argmin(np.abs(lats - pp_lat))])
nearest_lon = float(lons[np.argmin(np.abs(lons - pp_lon_adj))])

# Convert back to -180..180 for display if needed
disp_lon = nearest_lon
if disp_lon > 180:
    disp_lon = disp_lon - 360

points_df["glofas_lat"] = nearest_lat
points_df["glofas_lon"] = disp_lon

print(points_df)

# Interactive map (OpenStreetMap tiles; no token required)
fig = px.scatter_mapbox(
    points_df,
    lat="pour_lat", lon="pour_lon",
    hover_name="point_id",
    zoom=7,
    height=450,
)
fig.update_layout(mapbox_style="open-street-map", margin=dict(l=0,r=0,t=30,b=0))
fig.update_traces(marker=dict(size=12))
fig.update_layout(title="HydroBASINS pour point (virtual gauge origin)")
fig.show()

fig2 = go.Figure()
fig2.add_trace(go.Scattermapbox(
    lat=[pp_lat], lon=[pp_lon],
    mode="markers", marker=dict(size=12),
    name="Pour point",
    hovertext=[f"hybas_id={hybas_id}"]
))
fig2.add_trace(go.Scattermapbox(
    lat=[nearest_lat], lon=[disp_lon],
    mode="markers", marker=dict(size=12),
    name="Nearest GloFAS grid cell",
    hovertext=[f"grid lat={nearest_lat:.3f}, lon={disp_lon:.3f}"]
))
fig2.update_layout(mapbox_style="open-street-map", zoom=7, height=450, margin=dict(l=0,r=0,t=30,b=0),
                   title="Pour point mapped to nearest GloFAS grid cell (no averaging)")
fig2.show()


## Step 2 — Extract daily discharge time series from GRIB (if needed)

This notebook expects a **daily discharge time series** for the representative point.

If the cached file does not exist, we will:
1. scan the year folders under `glofas_grib_root`
2. open each yearly GRIB file with `xarray` + `cfgrib`
3. extract the nearest grid-cell discharge for the representative point
4. save a cached Parquet (or CSV fallback)

> This is intentionally simple and transparent. If performance becomes a concern later, we can convert the GRIB archive to chunked NetCDF/Zarr once and re-use it.


In [ ]:
# Extract or load cached daily discharge time series for this virtual gauge

index_dir = timeseries_out_dir / "_cfgrib_index"
index_dir.mkdir(parents=True, exist_ok=True)

def _open_grib_dataset(path: Path, idx_name: str) -> xr.Dataset:
    """Open GRIB robustly (handles multi-message files; keeps .idx in processed/)."""
    indexpath = str(index_dir / idx_name)
    try:
        return xr.open_dataset(path, engine="cfgrib", backend_kwargs={"indexpath": indexpath})
    except Exception:
        import cfgrib
        dsets = cfgrib.open_datasets(path, backend_kwargs={"indexpath": indexpath})
        # Prefer datasets with time + lat/lon
        for ds_ in dsets:
            coords = set(ds_.coords)
            if ('time' in coords or 'valid_time' in coords) and (('latitude' in coords or 'lat' in coords) and ('longitude' in coords or 'lon' in coords)):
                return ds_
        return dsets[0]

def _detect_discharge_var(ds: xr.Dataset) -> str:
    """Heuristic to pick the discharge variable from a GRIB Dataset."""
    if len(ds.data_vars) == 1:
        return list(ds.data_vars)[0]
    for name in ds.data_vars:
        if "dis" in name.lower() or "river" in name.lower() or "runoff" in name.lower():
            return name
    return list(ds.data_vars)[0]

def build_point_timeseries_from_grib(
    grib_root: Path,
    lat: float,
    lon: float,
    start: str,
    end: str,
) -> pd.Series:
    """Extract daily discharge at nearest grid cell for a given lat/lon from yearly GRIB files."""
    start_ts = pd.to_datetime(start)
    end_ts = pd.to_datetime(end)

    series_parts = []
    year_dirs = sorted([p for p in grib_root.glob('*') if p.is_dir() and p.name.isdigit()])
    if not year_dirs:
        raise FileNotFoundError(f"No year folders found under {grib_root}")

    for yd in year_dirs:
        year = int(yd.name)
        if year < start_ts.year or year > end_ts.year:
            continue

        files = sorted(yd.glob('*.grib*'))
        if not files:
            files = sorted(yd.rglob('*.grib*'))
        if not files:
            warnings.warn(f"No GRIB file found for year folder: {yd}")
            continue

        f = files[0]
        ds = _open_grib_dataset(f, idx_name=f"{f.stem}.idx")

        # Detect coordinate names
        lat_name = 'latitude' if 'latitude' in ds.coords else ('lat' if 'lat' in ds.coords else None)
        lon_name = 'longitude' if 'longitude' in ds.coords else ('lon' if 'lon' in ds.coords else None)
        if lat_name is None or lon_name is None:
            raise ValueError(f"Could not find lat/lon coords in {f}. Coords: {list(ds.coords)}")

        lons = ds[lon_name].values
        lon_adj = lon
        if np.nanmax(lons) > 180 and lon_adj < 0:
            lon_adj = lon_adj % 360

        v = _detect_discharge_var(ds)

        da = ds[v].sel({lat_name: lat, lon_name: lon_adj}, method='nearest')

        # Convert to pandas (handle different time coord conventions)
        if 'time' in da.coords:
            s = da.to_series()
            s.index = pd.to_datetime(s.index)
        elif 'valid_time' in da.coords:
            s = pd.Series(da.values, index=pd.to_datetime(da['valid_time'].values))
        else:
            raise ValueError(f"No time coordinate found in var '{v}'. dims={da.dims}, coords={list(da.coords)}")

        s = s.loc[(s.index >= start_ts) & (s.index <= end_ts)]
        series_parts.append(s)

    if not series_parts:
        raise RuntimeError("No data extracted. Check year folders, GRIB files, and date range.")

    out = pd.concat(series_parts).sort_index()
    out = out[~out.index.duplicated(keep='first')]
    return out.astype(float)

# Target cache file (one per L12)
cache_path_parquet = timeseries_out_dir / f"{hybas_id}__l12_discharge.parquet"
cache_path_csv = timeseries_out_dir / f"{hybas_id}__l12_discharge.csv"

if (not force_reextract_timeseries) and cache_path_parquet.exists():
    print("Loading cached time series:", cache_path_parquet)
    df = pd.read_parquet(cache_path_parquet)
    discharge_series = df['discharge'] if 'discharge' in df.columns else df.iloc[:, 0]
elif (not force_reextract_timeseries) and cache_path_csv.exists():
    print("Loading cached time series:", cache_path_csv)
    df = pd.read_csv(cache_path_csv, parse_dates=['time'])
    discharge_series = pd.Series(df['discharge'].values, index=df['time'])
else:
    if not HAS_CFGRIB:
        raise ImportError("cfgrib is required to extract GRIB time series. Install cfgrib + python-eccodes.")

    lat = float(points_df.loc[0, 'glofas_lat'])
    lon = float(points_df.loc[0, 'glofas_lon'])
    print(f"Extracting discharge for hybas_id={hybas_id} at lat={lat:.4f}, lon={lon:.4f}")

    discharge_series = build_point_timeseries_from_grib(
        glofas_grib_root, lat=lat, lon=lon, start=start_date, end=end_date
    )

    # Save cache
    try:
        discharge_series.to_frame('discharge').to_parquet(cache_path_parquet, index=True)
        print("Saved:", cache_path_parquet)
    except Exception as e:
        warnings.warn(f"Parquet write failed ({e}). Falling back to CSV.")
        out_df = discharge_series.to_frame('discharge').reset_index().rename(columns={'index':'time'})
        out_df.to_csv(cache_path_csv, index=False)
        print("Saved:", cache_path_csv)

discharge_series.name = "discharge"
print("Time series length:", len(discharge_series), "from", discharge_series.index.min(), "to", discharge_series.index.max())


## Step 3 — Explore the discharge time series (QA/QC)

Before EVT, check:
- time coverage and missingness
- obvious artefacts (constant segments, spikes)
- distribution shape (histogram + ECDF)

If the time series looks wrong, **stop here** and fix extraction/settings.


In [ ]:
# Basic QA/QC and exploratory plots

s = discharge_series.copy()
s = s.sort_index()

# Coverage + missingness
expected = pd.date_range(pd.to_datetime(start_date), pd.to_datetime(end_date), freq='D')
missing_days = expected.difference(s.index)
missing_pct = len(missing_days) / len(expected) * 100

print(f"Expected days: {len(expected):,}")
print(f"Observed days: {len(s):,}")
print(f"Missing days : {len(missing_days):,} ({missing_pct:.2f}%)")

# Flag constant segments (simple heuristic)
rolling_std = s.rolling(30, min_periods=10).std()
const_flag = (rolling_std < 1e-6).sum()
if const_flag > 0:
    print("Warning: detected near-constant 30-day windows. Inspect extraction and source files.")

# Time series plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=s.index, y=s.values, mode='lines', name='Discharge (m³/s)'))
fig.update_layout(
    title=f"Daily discharge — hybas_id={hybas_id} (virtual gauge)",
    xaxis_title="Date", yaxis_title="Discharge (m³/s)",
    height=380,
    margin=dict(l=40,r=10,t=50,b=40),
)
fig.show()

# Distribution plot
fig2 = make_subplots(rows=1, cols=2, subplot_titles=["Histogram", "ECDF"])
fig2.add_trace(go.Histogram(x=s.values, nbinsx=60, name='Histogram'), row=1, col=1)

# ECDF
x_sorted = np.sort(s.values)
y = np.arange(1, len(x_sorted)+1)/len(x_sorted)
fig2.add_trace(go.Scatter(x=x_sorted, y=y, mode='lines', name='ECDF'), row=1, col=2)

fig2.update_layout(height=380, showlegend=False, margin=dict(l=40,r=10,t=50,b=40))
fig2.update_xaxes(title_text="Discharge (m³/s)", row=1, col=1)
fig2.update_xaxes(title_text="Discharge (m³/s)", row=1, col=2)
fig2.update_yaxes(title_text="Count", row=1, col=1)
fig2.update_yaxes(title_text="P(X ≤ x)", row=1, col=2)
fig2.show()


## Step 4 — Threshold diagnostics (POT)

We start with a few high-quantile candidates (e.g., 95–99%) and use:
- **Mean Residual Life (MRL)**: should be roughly linear above the threshold
- **GPD parameter stability**: fitted parameters should not drift wildly

Choose:
- `threshold_m3s` (u)
- `run_length_days` (declustering window; default 5)


In [ ]:
# Candidate thresholds from quantiles + diagnostics (MRL + parameter stability)

series = discharge_series.dropna()

# Candidate thresholds from quantiles
cand_u = [float(series.quantile(q)) for q in candidate_quantiles]
cand_u = sorted(list(set(cand_u)))
print("Candidate thresholds (m³/s):", [round(u, 2) for u in cand_u])

# Suggest an expanded range for diagnostics
u_min, u_max = suggest_threshold_range(series, quantiles=(min(candidate_quantiles), max(candidate_quantiles)))
print(f"Suggested diagnostic range: {u_min:.2f} to {u_max:.2f} m³/s")

# Build a threshold grid for diagnostics
u_grid = np.linspace(u_min, u_max, 20)

mrl_df = mean_residual_life(series, thresholds=u_grid)
stab_df = gpd_parameter_stability(series, thresholds=u_grid, min_exceedances=30)

# Plot MRL
fig = go.Figure()
fig.add_trace(go.Scatter(x=mrl_df['threshold'], y=mrl_df['mean_excess'], mode='lines+markers', name='Mean excess'))
fig.update_layout(
    title="Mean Residual Life (MRL) — look for approx. linear region",
    xaxis_title="Threshold u (m³/s)",
    yaxis_title="Mean exceedance (m³/s)",
    height=380, margin=dict(l=40,r=10,t=50,b=40)
)
fig.show()

# Plot stability
fig2 = make_subplots(rows=1, cols=2, subplot_titles=["GPD shape ξ", "GPD scale σ"], shared_xaxes=True)
fig2.add_trace(go.Scatter(x=stab_df['threshold'], y=stab_df['shape_xi'], mode='lines+markers', name='ξ'), row=1, col=1)
fig2.add_trace(go.Scatter(x=stab_df['threshold'], y=stab_df['scale_sigma'], mode='lines+markers', name='σ'), row=1, col=2)
fig2.update_layout(
    title="GPD parameter stability — avoid thresholds where parameters drift strongly",
    height=420, showlegend=False, margin=dict(l=40,r=10,t=60,b=40)
)
fig2.update_xaxes(title_text="Threshold u (m³/s)", row=1, col=1)
fig2.update_xaxes(title_text="Threshold u (m³/s)", row=1, col=2)
fig2.update_yaxes(title_text="ξ", row=1, col=1)
fig2.update_yaxes(title_text="σ", row=1, col=2)
fig2.show()


## Step 5 — Outputs to copy into YAML

We compute:
- `threshold_m3s` (selected)
- `run_length_days` (selected)
- `event_rate_per_year` = **λ** (Poisson rate of independent exceedances/year after declustering)

Copy/paste the snippet printed at the end into your basin YAML under `evt:`.


In [ ]:
# Choose threshold + decluster exceedances, then estimate Poisson λ (events/year)

# --- Choose a threshold ---
# Option A: manual (recommended for auditability)
threshold_m3s = cand_u[-1]  # default to highest quantile candidate; adjust after reviewing plots
run_length_days = run_length_days_default

print(f"Selected threshold_m3s = {threshold_m3s:.2f}")
print(f"Selected run_length_days = {run_length_days}")

# Extract declustered POT peaks
peaks_df = extract_declust_pot(series, threshold=threshold_m3s, run_length_days=run_length_days)

print("Independent peaks extracted:", len(peaks_df))
display(peaks_df.head(10)) if 'display' in globals() else peaks_df.head(10)

# Plot peaks on time series (last 5 years for readability)
end_plot = series.index.max()
start_plot = end_plot - pd.Timedelta(days=365*5)
mask = (series.index >= start_plot)

fig = go.Figure()
fig.add_trace(go.Scatter(x=series.index[mask], y=series.values[mask], mode='lines', name='Discharge'))
fig.add_hline(y=threshold_m3s, line_dash='dash', annotation_text='threshold u', annotation_position='top left')

peaks_recent = peaks_df[(peaks_df['time'] >= start_plot) & (peaks_df['time'] <= end_plot)]
fig.add_trace(go.Scatter(
    x=peaks_recent['time'], y=peaks_recent['peak'],
    mode='markers', name='Declustered peaks',
    marker=dict(size=8)
))
fig.update_layout(
    title="Declustered peaks above threshold (recent years)",
    xaxis_title="Date", yaxis_title="Discharge (m³/s)",
    height=420, margin=dict(l=40,r=10,t=60,b=40)
)
fig.show()

# Annual exceedance counts → Poisson λ
peaks_df['year'] = peaks_df['time'].dt.year
annual_counts = peaks_df.groupby('year').size()

# Ensure we include all years in range (missing years → 0)
years = pd.Index(range(pd.to_datetime(start_date).year, pd.to_datetime(end_date).year + 1), name='year')
annual_counts = annual_counts.reindex(years, fill_value=0)

lambda_events_per_year = float(annual_counts.mean())
var_counts = float(annual_counts.var(ddof=1)) if len(annual_counts) > 1 else 0.0

print("\nAnnual exceedance count summary")
print("Mean (λ):", round(lambda_events_per_year, 3), "events/year")
print("Variance:", round(var_counts, 3))
if lambda_events_per_year > 0:
    print("Var/Mean:", round(var_counts / lambda_events_per_year, 3))

fig3 = go.Figure()
fig3.add_trace(go.Bar(x=annual_counts.index.astype(str), y=annual_counts.values))
fig3.update_layout(
    title="Annual independent exceedances (post-declustering)",
    xaxis_title="Year", yaxis_title="# exceedances",
    height=380, margin=dict(l=40,r=10,t=60,b=40)
)
fig3.show()

# YAML snippet (safe copy/paste)
print("\n--- Copy/paste into your basin YAML under evt: ---\n") 
print("evt:") 
print(f"  method: POT-GPD") 
print(f"  threshold_m3s: {threshold_m3s:.6g}") 
print(f"  run_length_days: {int(run_length_days)}") 
print(f"  event_rate_per_year: {lambda_events_per_year:.6g}") 
print("  # gpd_shape_xi and gpd_scale_sigma will be filled in the next notebook") 


In [ ]:
from datetime import datetime
import json

analyst_name = "YOUR_NAME"
decision_notes = "Short rationale for chosen threshold/run-length. Mention any data issues."

decision = {
    "hybas_id": hybas_id,
    "basin_id": basin_cfg.basin_id,
    "start_date": start_date,
    "end_date": end_date,
    "threshold_m3s": float(threshold_m3s),
    "run_length_days": int(run_length_days),
    "event_rate_per_year": float(lambda_events_per_year),
    "analyst_name": analyst_name,
    "decision_notes": decision_notes,
    "timestamp_utc": datetime.utcnow().isoformat() + "Z",
}

out_dir = REPO_ROOT / "calibration" / "outputs" / "evt_threshold_decisions"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"{hybas_id}__evt_threshold_decision.json"

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(decision, f, indent=2)

print("Saved decision log:", out_path)
print(json.dumps(decision, indent=2)[:1000])


## Decision log (recommended)

Record what you decided and why. This helps with peer review and later reproducibility.

- analyst name / initials
- date
- chosen threshold and run-length
- any anomalies you observed
- link to issue / ticket (optional)


## Next steps

Proceed to the next calibration notebook to fit the **GPD parameters** (shape ξ and scale σ) and to generate synthetic events. Keep a short decision log (why you chose this threshold, any anomalies, and any notes for peer review).